# MAREA and NIDEM against the number of scenes, at one site

**The question** (meeting of 2026-10-20): *MAREA one curve, NIDEM another, and see how each one
responds to being given different numbers of images.*

N Sentinel-2 scenes are drawn at random from the site's whole 2023–2025 archive (the pool **all**:
every cube scene with an overpass time and a finite boundary level; clouds are masked pixel by
pixel). On every draw both methods run as published:

* **MAREA** is re-fitted from scratch: one lag per band of distance from the mouth, then the
  per-pixel inversion, with no quality rule;
* **NIDEM** runs its own recipe on the same scenes: ITEM's pixel mask, 9 tide intervals, median
  NDWI composites, waterline contours, TIN and the confidence filter.

Both are scored on one fixed set of points: the 194 RTK points at Villaviciosa, a seeded sample
of 100 000 Vaklodingen pixels at the Dutch sites.

This is the experiment of `experiments/v14_scene_count_curves.py` (MAREA, `--lags refit`) and
`experiments/v15_nidem_scene_curves.py` (NIDEM), written as plain calls to `pyintertidal`.
Section 10 checks that the numbers equal those of the server run. The four sites are drawn
together by `03b_marea_vs_nidem_summary.ipynb`.

## Parameters

`SITE` picks the site; `run_03_on_server.sh` sets it through the environment variable `NB_SITE`.
The N grid and the replicates default to v14's plan for the site
(`results/scene_count_curves/<site>_refit_settings.json`); `NB_N_GRID` and `NB_REPS` shorten
them for a test.

In [ ]:
import os

SITE = os.environ.get("NB_SITE", "escalda")       # villaviciosa | escalda | wadden | ems

# v14's plan for the pool 'all': the N values (the last one is the whole pool),
# the replicates per N up to 100 and the replicates per N above 100
PLAN = {
    "villaviciosa": ([10, 15, 20, 30, 40, 60, 80, 100, 150, 200, 300, 400, 465], 10, 5),
    "escalda":      ([10, 15, 20, 30, 40, 60, 80, 100, 150, 200, 300, 400, 464], 5, 3),
    "wadden":       ([10, 15, 20, 30, 40, 60, 80, 100, 150, 200, 300, 400, 462], 5, 3),
    "ems":          ([10, 20, 40, 80, 150, 300, 462], 3, 2),
}
N_GRID, REPS, REPS_LARGE = PLAN[SITE]
if os.environ.get("NB_N_GRID"):                   # e.g. "40,100" for a quick test
    N_GRID = [int(v) for v in os.environ["NB_N_GRID"].split(",")]
if os.environ.get("NB_REPS"):
    REPS = REPS_LARGE = int(os.environ["NB_REPS"])

SEED        = 20261019                            # v14's seed, so the draws are the same
MAX_UNITS   = 100_000                             # scored pixels at the Dutch sites
TIME_EXTENT = ("2023-01-01", "2025-12-31")
OUT_DIR     = os.environ.get("NB_OUT", "results/scene_count_curves_nb")
SERVER_RUN  = "results/scene_count_curves"        # v14 and v15 CSVs, for section 10

print(f"{SITE}: N = {N_GRID}, {REPS} replicates (N > 100: {REPS_LARGE}) -> {OUT_DIR}/{SITE}.csv")

## 0 · Setup

The notebook runs from the repository root, so the `pyintertidal`, `configs/` and data used are
the repository's. Two BLAS threads, as in the server run of v14.

In [ ]:
import json, sys, time, zlib

for _ in range(3):                                # this notebook lives in notebooks/validation/
    if os.path.isdir("pyintertidal"):
        break
    os.chdir("..")
sys.path.insert(0, os.getcwd())
for v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ.setdefault(v, "2")                 # before numpy loads its BLAS

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import ScalarFormatter
import yaml

import pyintertidal as pit
from pyintertidal import marea, rtk
from pyintertidal import tide_estimators as te
from pyintertidal import waterline_contour as wc
from pyintertidal.boundary import make_boundary, PyTMDBoundary
from pyintertidal.elevation_validation import ols_against_truth

os.makedirs(OUT_DIR, exist_ok=True)
print("repository", os.getcwd(), "· pyintertidal", pit.__version__)

## 1 · Study area and the published 10 m run

The inputs are those of the published 10 m products: MAREA's extraction (the NDWI and the
clear-sky flag, SCL 4–7, of every intertidal pixel in every cube scene), MAREA's product (its
bands of distance from the mouth) and the cube that NIDEM reads.

In [ ]:
SITES = {
    "villaviciosa": dict(products="products_marea", marea="products_marea/marea_eot20_g0_2023-2025",
                         cube="ndwi_cube_villaviciosa_grande_10y.nc"),
    "escalda": dict(products="products_escalda_10m", marea="products_escalda_10m/marea_eot20_g2",
                    comparison="products_escalda_10m/comparison_eot20_g2",
                    cube="ndwi_cube_escalda_2023-2025_10m.nc", n_gauges=2, held_out=["trnz"]),
    "wadden": dict(products="products_wadden_10m", marea="products_wadden_10m/marea_eot20_g1",
                   comparison="products_wadden_10m/comparison_eot20_g1",
                   cube="ndwi_cube_wadden_2023-2025_10m.nc", n_gauges=1, held_out=["harl"]),
    "ems": dict(products="products_ems_10m", marea="products_ems_10m/marea_eot20_g1",
                comparison="products_ems_10m/comparison_eot20_g1",
                cube="ndwi_cube_ems_2023-2025_10m.nc", n_gauges=1, held_out=["delf"]),
}
S = SITES[SITE]
aoi = pit.sites.get(SITE)
print(aoi)

In [ ]:
with np.load(f"{S['products']}/marea_extract.npz") as ex:
    keep = ex["keep"]                             # the intertidal pixels (flat indices of the frame)
    ext_dates = ex["dates"].astype(str)           # one date per cube scene
    H, W = (int(v) for v in ex["shape"])
mp = np.load(f"{S['marea']}/marea.npz")           # the published MAREA product
result = json.load(open(f"{S['marea']}/result.json", encoding="utf-8"))
print(f"{len(ext_dates)} cube scenes ({ext_dates[0]} .. {ext_dates[-1]}), frame {H} x {W}, "
      f"{len(keep):,} intertidal px, {len(result['centros_km'])} bands of distance from the mouth")

## 2 · The pool of scenes

The boundary is the one each published run used: EOT20 at Villaviciosa; EOT20 blended with the
nearest IOC gauges at the Dutch sites, the in-estuary gauge held out. The pool **all** is every
cube scene of 2023–2025 with an overpass time and a finite boundary level there: NIDEM's
published record, and the pool v14 drew from.

In [ ]:
if SITE == "villaviciosa":
    lat_c, lon_c = aoi.centroid
    boundary = PyTMDBoundary("EOT20", lat_c, lon_c, directory="tide_models")
else:
    pit.net.use_system_certificates()
    boundary = make_boundary(aoi, tide_model="EOT20", n_gauges=S["n_gauges"],
                             period=TIME_EXTENT, exclude=S["held_out"])

overpass = json.load(open(f"{S['products']}/overpass_times.json"))
pool = np.flatnonzero([d in overpass and TIME_EXTENT[0] <= d <= TIME_EXTENT[1] for d in ext_dates])
t_pool = pd.DatetimeIndex(pd.to_datetime([overpass[d] for d in ext_dates[pool]])).tz_localize(None)
h_pool = np.asarray(boundary.levels(t_pool), float)
ok = np.isfinite(h_pool)
pool, t_pool, h_pool = pool[ok], t_pool[ok], h_pool[ok]     # pool = scene positions in cube and extraction
print(f"pool 'all': {len(pool)} scenes · boundary {getattr(boundary, 'name', type(boundary).__name__)}")

fig, ax = plt.subplots(figsize=(12, 3))
ax.plot(t_pool, h_pool, ".", ms=4, color="0.35")
ax.set_ylabel("boundary level (m)")
ax.set_title(f"The {len(pool)} scenes of the pool, at their tide level", loc="left")
ax.grid(alpha=0.3);

## 3 · The fixed set of scored points

Every draw is scored on the same units, chosen once:

* **Villaviciosa**: the RTK-GNSS points read with `pyintertidal.rtk.load_rtk()` and its default
  arguments (the development points), on the intertidal pixels where the published MAREA has an
  elevation: 194 points on 120 pixels, heights in the Alicante datum;
* **Dutch sites**: the Rijkswaterstaat Vaklodingen (latest survey of each 20 m cell) at every
  intertidal pixel where the published MAREA has an elevation, kept inside the published maps'
  tide range ± 0.25 m (the rule of the comparison notebooks); then a seeded sample of 100 000 of
  those pixels with v14's seed.

In [ ]:
if SITE == "villaviciosa":
    z_published = mp["z"].astype(float)
    dev = rtk.load_rtk()                                       # default arguments only
    flat = np.asarray(dev["row"], np.int64) * W + np.asarray(dev["col"], np.int64)
    pos = np.clip(np.searchsorted(keep, flat), 0, len(keep) - 1)   # each point's intertidal pixel
    z_rtk = np.asarray(dev["elev_alicante"] if "elev_alicante" in dev else dev["elev"], float)
    F = np.flatnonzero((keep[pos] == flat) & np.isfinite(z_rtk) & np.isfinite(z_published[pos]))
    units, truth = pos[F], z_rtk[F]
else:
    import pyproj, xarray as xr
    from experiments import v1_vaklodingen_validate as v1     # the comparison notebooks' Vaklodingen reader
    pub = np.load(f"{S['comparison']}/products.npz")           # the published no-delay and MAREA maps
    z_published = pub["z_marea"].astype(float)
    with xr.open_dataset(S["cube"]) as ds:
        xs, ys = ds["x"].values.astype(float), ds["y"].values.astype(float)
        crs = pyproj.CRS.from_wkt(ds["crs"].attrs.get("crs_wkt") or ds["crs"].attrs.get("spatial_ref"))
    xq, yq = pyproj.Transformer.from_crs(crs, "EPSG:28992", always_xy=True).transform(xs[keep % W], ys[keep // W])
    truth_px, _ = v1.sample_truth(v1.truth_grid(SITE), xq, yq)
    lo, hi = float(np.nanmin(pub["z_plain"])), float(np.nanmax(pub["z_plain"]))
    truth_px = np.where((truth_px >= lo - 0.25) & (truth_px <= hi + 0.25), truth_px, np.nan)
    F = np.flatnonzero(np.isfinite(truth_px) & np.isfinite(z_published))
    rng = np.random.default_rng(np.random.SeedSequence([SEED, zlib.crc32(SITE.encode()) & 0x7FFFFFFF, 7]))
    pick = np.sort(rng.permutation(len(F))[:MAX_UNITS])        # v14's seeded sample
    units, truth = F[pick], truth_px[F][pick]

In [ ]:
units_flat = keep[units]                  # frame index of every unit (NIDEM's maps are full frames)
px = np.unique(units)                     # the intertidal pixels MAREA inverts
unit_col = np.searchsorted(px, units)     # each unit's column among them
print(f"{len(units):,} scored units on {len(px):,} pixels ({len(F):,} eligible) · "
      f"truth {np.nanmin(truth):+.2f} .. {np.nanmax(truth):+.2f} m")

## 4 · MAREA's record: NDWI and clear-sky flags of the pool

Two sets of columns of the extraction are kept, for the scenes of the pool: the pixels to invert
(above) and the pixels of the lag fit. The lag fit uses at most 1 200 pixels per band, the seeded
draw `marea.reconstruct` makes (`configs/m2.yaml`), so every refit sees the pixels of the
published fit.

In [ ]:
M2 = yaml.safe_load(open("configs/m2.yaml", encoding="utf-8"))
MAX_PX_BAND = M2["max_px_banda"]["m2a"]
centers = np.asarray(result["centros_km"], float)      # band centres (km from the mouth)
BAND_KM = float(mp["band_km"][0])
band_all = mp["band"].astype(int)                      # band of every intertidal pixel

rng = np.random.default_rng(M2["seed"] + 31)
fit_px = []
for k in range(len(centers)):
    c = np.flatnonzero(band_all == k)
    fit_px.append(np.sort(rng.choice(c, MAX_PX_BAND, replace=False)) if len(c) > MAX_PX_BAND else c)
band_fit = np.concatenate([np.full(len(c), k) for k, c in enumerate(fit_px)])
fit_px = np.concatenate(fit_px)

# distance from the mouth of the pixels to invert (the published comparison's copy at the Dutch sites)
s_km = (mp["s_km"] if SITE == "villaviciosa" else pub["s_km"]).astype(float)[px]

In [ ]:
t0 = time.time()
with np.load(f"{S['products']}/marea_extract.npz") as ex:
    Y_all, C_all = ex["Y"], ex["C"]                  # (cube scenes, intertidal px): a few GB at the Dutch sites
Y_px, C_px = np.nan_to_num(Y_all[np.ix_(pool, px)]), C_all[np.ix_(pool, px)]
Y_fit, C_fit = np.nan_to_num(Y_all[np.ix_(pool, fit_px)]), C_all[np.ix_(pool, fit_px)]
del Y_all, C_all
print(f"{len(pool)} scenes x ({len(px):,} px to invert + {len(fit_px):,} lag-fit px in "
      f"{len(centers)} bands) [{time.time() - t0:.0f} s]")

## 5 · The boundary at every lag

MAREA reads the boundary τ minutes before each overpass: the lag fit at the shifts of
`marea.BANK_TAUS` (−45 to 180 min, every 15), the inversion at every whole minute a pixel can
receive (0 to 180). All of them come from one call over the shifted instants, as in v14.
`levels[0]` is the level at the overpass, `h_pool`.

In [ ]:
LAGS = sorted({int(t) for t in marea.BANK_TAUS} | set(range(181)))
shifted = {tau: t_pool - pd.Timedelta(minutes=tau) for tau in LAGS}
instants = pd.DatetimeIndex(np.unique(np.concatenate([s.values for s in shifted.values()])))
t0 = time.time()
lev = np.asarray(boundary.levels(instants), float)
levels = {tau: lev[np.searchsorted(instants.values, s.values)] for tau, s in shifted.items()}
assert np.array_equal(levels[0], h_pool)
print(f"{len(LAGS)} lags x {len(pool)} scenes from one call over {len(instants):,} instants "
      f"[{time.time() - t0:.0f} s]")

## 6 · MAREA on N scenes

Three steps, each the published one:

1. **one lag per band** of distance from the mouth, by profiled Bernoulli likelihood of the
   wet/dry record (`te.m2a_rasch` with the settings of `marea.reconstruct`); a negative lag is
   clipped to 0, since a delay cannot be negative;
2. **the lag of every pixel**: the band lags smoothed along the distance from the mouth, to the
   whole minute;
3. **the inversion** (`marea.invert_series`) of every pixel with the boundary read that many
   minutes earlier, over the draw's own tide range, with no quality rule (`min_bracket=0`, as the
   published products).

In [ ]:
FIT_KW = dict(sigma0=M2["sigma0_m"], sg_grid=tuple(M2["sigma_perfil_m"]), tau_bounds=marea.TAU_BOUNDS,
              z_points=M2["z_puntos"], max_px_band=MAX_PX_BAND, monotone=True)

def marea_on(idx):
    """MAREA on the pool scenes idx: the elevation of every pixel of px, and the band lags."""
    # 1. one lag per band
    wet = C_fit[idx] & (Y_fit[idx] > 0)
    bank = te.ShiftBank(marea.BANK_TAUS, [levels[int(t)][idx] for t in marea.BANK_TAUS])
    fit = te.m2a_rasch(wet, C_fit[idx], bank, band_fit, centers, len(centers),
                       rng=np.random.default_rng(M2["seed"] + 31), **FIT_KW)
    tau = np.asarray(fit["tau"], float)
    tau = np.where(np.abs(tau) > marea.TAU_DETECT_MIN, tau, 0.0)
    tau = np.where(tau < 0, 0.0, tau)                       # causality
    # 2. the lag of every pixel, to the minute
    tau_px = (te.smooth_lag_profile(centers, tau, s_km, bandwidth_km=BAND_KM)
              if len(centers) > 1 else np.zeros(len(px)))
    tau_px = np.where(np.isfinite(tau_px), np.round(tau_px), 0.0)
    # 3. the inversion, pixels grouped by their lag
    lo, hi = float(levels[0][idx].min()), float(levels[0][idx].max())
    z = np.full(len(px), np.nan)
    for t in np.unique(tau_px):
        cols = np.flatnonzero(tau_px == t)
        z[cols], _ = marea.invert_series(Y_px[np.ix_(idx, cols)], C_px[np.ix_(idx, cols)],
                                         levels[int(t)][idx], lo, hi, min_bracket=0)
    return z, tau

## 7 · NIDEM on N scenes

NIDEM's published recipe (`pyintertidal.waterline_contour`, as
`experiments/v7_waterline_contour.py` runs it): the draw's scenes go into 9 intervals of their own
observed tide range; per interval, the median NDWI of every pixel over its valid observations
(ITEM's pixel mask), read from the cube over the full frame; then the relative-extents map R, its
waterline contours tagged with each interval's median level, a TIN between them, kept where R is
1–8 and the NDWI confidence is at most 0.25 (*filtered*, the validated product).

As in production a draw is **not computable** when an interval holds no scene, or when fewer than
1 % of the intertidal pixels have a valid observation in all 9 intervals (the pre-flight guard).
It then has coverage 0 and no error.

In [ ]:
def nidem_on(idx):
    """NIDEM (filtered) on the pool scenes idx, at the scored units; None when not computable."""
    try:
        j, edges, Z, U, sizes = wc.interval_assignment(h_pool[idx])
    except wc.EmptyIntervalError:
        return None, "not_computable_empty_interval"
    composites = wc.stream_composites(S["cube"], pool[idx], j, rule="item", verbose=False)
    cov = wc.interval_coverage(composites["count"].reshape(wc.N_INTERVALS, -1)[:, keep], raise_error=False)
    if not cov["passed"]:
        return None, "not_computable_preflight"
    res = wc.nidem(composites, Z, U)
    return res["filtered"].reshape(-1)[units_flat], "ok"

## 8 · The draws

A draw takes N scenes of the pool at random. It is v14's draw: a pure function of (seed, site,
pool, N, replicate), so these are the scenes of the server run; N = the whole pool is one run.
Both methods are scored on the units of section 3:

* `rmse`: root mean square of product − truth;
* `bias`: median of product − truth, the datum offset;
* `rmse_centred`: the RMSE once each one's median is removed, the published error
  (`ols_against_truth`; at least 20 RTK points or 30 pixels);
* `coverage`: share of the units with an estimate.

One row per (N, replicate, method) goes to `<OUT_DIR>/<site>.csv` as each draw finishes.

In [ ]:
def draw(N, rep):
    """v14's draw: the sorted pool positions of replicate rep at size N."""
    if N >= len(pool):
        return np.arange(len(pool))
    seq = np.random.SeedSequence([SEED, zlib.crc32(SITE.encode()) & 0x7FFFFFFF, 2, N, rep])  # 2 = pool 'all'
    return np.sort(np.random.default_rng(seq).choice(len(pool), N, replace=False))

def score(p):
    """The errors of one product at the scored units (p None: not computable)."""
    if p is None:
        return dict(rmse=np.nan, bias=np.nan, rmse_centred=np.nan, coverage=0.0)
    p = np.asarray(p, float)
    f = ols_against_truth(truth, p, min_n=20 if SITE == "villaviciosa" else 30)
    m = np.isfinite(p) & np.isfinite(truth)
    return dict(rmse=float(np.sqrt(np.mean((p[m] - truth[m]) ** 2))) if f else np.nan,
                bias=f["bias_m"] if f else np.nan, rmse_centred=f["rmse_m"] if f else np.nan,
                coverage=float(np.isfinite(p).mean()))

jobs = [(N, rep) for N in N_GRID for rep in range(1 if N >= len(pool) else REPS_LARGE if N > 100 else REPS)]
print(f"{len(jobs)} draws")

In [ ]:
CSV = f"{OUT_DIR}/{SITE}.csv"
rows = []
for N, rep in jobs:
    idx = draw(N, rep)
    t0 = time.time()
    z, tau = marea_on(idx)
    a = dict(site=SITE, method="MAREA", N=len(idx), replicate=rep, status="ok",
             **score(z[unit_col]), seconds=round(time.time() - t0, 2))
    t0 = time.time()
    p, status = nidem_on(idx)
    b = dict(site=SITE, method="NIDEM", N=len(idx), replicate=rep, status=status,
             **score(p), seconds=round(time.time() - t0, 2))
    rows += [a, b]
    pd.DataFrame(rows).to_csv(CSV, index=False)
    print(f"N={len(idx):4d} rep {rep}: MAREA {a['rmse_centred']:.3f} m, cov {a['coverage']:.0%} "
          f"[{a['seconds']:.0f} s] | NIDEM {b['rmse_centred']:.3f} m, cov {b['coverage']:.0%} "
          f"[{b['seconds']:.0f} s] {'' if status == 'ok' else status}")
print(f"-> {CSV}")

## 9 · The curves

Median over the replicates that have a score, and the 10–90 % range across them. A
not-computable NIDEM draw has no score; the table says how many draws each N had and how many
NIDEM could compute.

In [ ]:
df = pd.read_csv(CSV)
summary = (df.groupby(["N", "method"])
             .agg(draws=("replicate", "size"), computable=("status", lambda s: int((s == "ok").sum())),
                  rmse_centred=("rmse_centred", "median"), bias=("bias", "median"),
                  coverage=("coverage", "median"), seconds=("seconds", "median"))
             .unstack("method"))
summary.round(3)

In [ ]:
STYLE = {"MAREA": dict(color="#0A7FA8", marker="s"), "NIDEM": dict(color="#B7791F", marker="o")}
fig, axes = plt.subplots(1, 2, figsize=(13, 4.6))
for method, st in STYLE.items():
    g = df[df.method == method].groupby("N")
    for ax, col in zip(axes, ("rmse_centred", "bias")):
        med = g[col].median().dropna()
        ax.fill_between(med.index, g[col].quantile(0.1).loc[med.index], g[col].quantile(0.9).loc[med.index],
                        color=st["color"], alpha=0.15, lw=0)
        ax.plot(med.index, med.values, "-", lw=2, ms=6, label=method, **st)
axes[0].set_ylabel("centred RMSE (m)")
axes[1].set_ylabel("bias: median(product − truth) (m)")
axes[1].axhline(0, color="0.5", lw=0.8)
for ax in axes:
    ax.set_xscale("log"); ax.set_xticks(sorted(df.N.unique()))
    ax.xaxis.set_major_formatter(ScalarFormatter()); ax.minorticks_off()
    ax.set_xlabel("scenes drawn from the pool (N)"); ax.grid(alpha=0.3); ax.legend(frameon=False)
truth_name = "194 RTK points" if SITE == "villaviciosa" else f"{len(units):,} Vaklodingen pixels"
fig.suptitle(f"{aoi.name}: error against the number of scenes ({truth_name})", x=0.01, ha="left")
fig.tight_layout()
fig.savefig(f"{OUT_DIR}/{SITE}.png", dpi=150, bbox_inches="tight")

## 10 · Check against the server run

The server run wrote MAREA's rows to `<site>_refit.csv` (v14; pool `all`, no quality rule:
`mb` 0) and NIDEM's to `<site>_nidem_all.csv` (v15); their `rmse` is the centred RMSE. For every
(N, replicate, method) in both runs the centred RMSE, the bias and the coverage must agree within
1e-9 m, and NIDEM's status must be the same.

In [ ]:
v14 = pd.read_csv(f"{SERVER_RUN}/{SITE}_refit.csv").query("pool == 'all' and mb == 0").assign(method="MAREA", status="ok")
v15 = pd.read_csv(f"{SERVER_RUN}/{SITE}_nidem_all.csv").assign(method="NIDEM")
server = pd.concat([v14, v15])[["method", "N", "replicate", "status", "rmse", "bias", "coverage"]]
both = df.merge(server.rename(columns={"rmse": "rmse_centred"}), on=["method", "N", "replicate"],
                suffixes=("", "_server"))

worst = {}
for col in ("rmse_centred", "bias", "coverage"):
    a, b = both[col], both[f"{col}_server"]
    assert (a.isna() == b.isna()).all(), f"{col}: a value missing in one run only"
    worst[col] = float(np.nan_to_num((a - b).abs().max()))
print(f"{len(both)} of {len(df)} rows are in the server run; largest differences: {worst}")
assert len(both) and (both.status == both.status_server).all() and max(worst.values()) <= 1e-9
both[["method", "N", "replicate", "status", "rmse_centred", "rmse_centred_server", "bias", "bias_server",
      "coverage", "coverage_server"]]